# LoanLens - Data Cleaning

## Objective

The goal of this notebook is to clean the errors we found in the eda notebook where we made the analysis this will be a transformation notebook.

Cleaning will be performed in sequence so that we don't miss anything and preserve useful information for machine learning.

Dataset : application_train.csv

Expected Output : application_train_clean.csv


## Cleaning Plan
| Problem ID | Problem | Severity | Decision | Status |
|------------|----------|----------|----------|--------|
| P1 | Duplicate Rows | Low | Check & Remove | ⏳. |
| P2 | Missing Values | High | Analyze column-wise | ⏳ ..|
| P3 | Sentinel Values | High | Replace invalid values | ⏳ ...|
| P4 | Outliers | Medium | Decide treatment | ⏳ ....|
| P5 | Data Types | Low | Verify | ⏳..... |
| P6 | Constant Columns | Low | Remove if needed | ⏳...... |
| P7 | High Missing Columns | High | Decide drop/impute | ⏳....... |
| P8 | Redundant Features | Medium | Review | ⏳ ........|
| P9 | Target Leakage | High | Verify | ⏳......... |

# P1 first why are duplicate rows important 

### Why are duplicate rows important?

Duplicate records can bias statistical analysis and machine learning models by giving certain observations more importance than others.

During EDA, duplicate records were not explicitly removed, therefore they are verified before any further preprocessing.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("../data/application_train.csv")

print(f"Dataset Shape : {df.shape}")

Dataset Shape : (307511, 122)


In [2]:
duplicate_rows = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


### Observation

No duplicate records were found in the dataset.

### Cleaning Decision

No duplicate removal was required. The original dataset is retained without modification.
For a better understanding we performed it here as well it was already done in eda nootebook as well.

# P2 - Sentinel Values

## Problem

Some datasets use placeholder values to represent missing or unknown information instead of actual null values.

During the EDA phase, the `DAYS_EMPLOYED` feature was found to contain the value `365243`, which does not represent a realistic number of employment days.

Keeping this value would distort statistical analysis and negatively affect model training.

Therefore, these values will be replaced while preserving the information that they originally contained.

Let's see the verification below 

In [15]:
anomaly_count = (df['DAYS_EMPLOYED'] == 365243).sum()
print(f"Anomalous rows: {anomaly_count} ({anomaly_count/len(df)*100:.2f}%)")


Anomalous rows: 0 (0.00%)


In [11]:
# create an anamoly flag is to preserve the information

df["DAYS_EMPLOYED_ANOM"] = ( df["DAYS_EMPLOYED"] == 365243 )

df["DAYS_EMPLOYED_ANOM"].value_counts()

/var/folders/3d/_njm_nr56yj72jj4qzhn0ctc0000gp/T/ipykernel_15630/1482707741.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DAYS_EMPLOYED_ANOM"] = ( df["DAYS_EMPLOYED"] == 365243 )


DAYS_EMPLOYED_ANOM
False    252137
True      55374
Name: count, dtype: int64

In [12]:
df["DAYS_EMPLOYED"] = (df["DAYS_EMPLOYED"].replace(365243, np.nan)) #replacing the values with nan.

In [13]:
print("Remaining anomalous values:", (df["DAYS_EMPLOYED"] == 365243).sum())

df["DAYS_EMPLOYED"].describe()

Remaining anomalous values: 0


count   252137.000
mean     -2384.169
std       2338.360
min     -17912.000
25%      -3175.000
50%      -1648.000
75%       -767.000
max          0.000
Name: DAYS_EMPLOYED, dtype: float64

Over here when we used describe we found the employee day had exceptionally higher days which is not possible 